# Garbage Classification based on text
### ENSF 617 - A2

Transfer learning with pretrained model [Distilbert](https://medium.com/huggingface/distilbert-8cf3380435b5)-base-uncased

Imports

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import os
import re
import numpy as np
import matplotlib.pyplot as plt

d:\conda_envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Configurations

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
NUM_CLASSES = 4
BATCH_SIZE = 32
EPOCHS = 10
UNFREEZE_EPOCH = 5
INITIAL_LR = 1e-3
FINE_TUNE_LR = 2e-5
MAX_LEN = 128

DATA_ROOT = "E:\\pigeon\\Documents\\ENSF617A2\\garbage_data\\CVPR_2024_dataset_"
SAVE_PATH = "best_text_model.pth"

Load Tokenizer and Base Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(device)

Read Text Data from given Folder 

Assuming your folder looks like this
```
root/
   Train/
      Blue/
      Green/
      ...
```

In [ ]:
def read_text_files_with_labels(path):
    texts = []
    labels = []

    class_folders = sorted([
        f for f in os.listdir(path)
        if os.path.isdir(os.path.join(path, f)) and not f.startswith(".")
    ])

    label_map = {class_name: idx for idx, class_name in enumerate(class_folders)}

    # extract text from file names
    for class_name in class_folders:
        class_path = os.path.join(path, class_name)

        for file_name in sorted(os.listdir(class_path)):
            file_path = os.path.join(class_path, file_name)

            if os.path.isfile(file_path):
                text, _ = os.path.splitext(file_name)
                text = text.replace("_", " ")
                text = re.sub(r"\d+", "", text)

                texts.append(text)
                labels.append(label_map[class_name])

    return np.array(texts), np.array(labels)

Load Data

In [ ]:
train_texts, train_labels = read_text_files_with_labels(os.path.join(DATA_ROOT, "Train"))
val_texts, val_labels = read_text_files_with_labels(os.path.join(DATA_ROOT, "Val"))
test_texts, test_labels = read_text_files_with_labels(os.path.join(DATA_ROOT, "Test"))

Define Dataset Class

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

Data Loaders

In [ ]:
train_ds = TextDataset(train_texts, train_labels, tokenizer)
val_ds   = TextDataset(val_texts, val_labels, tokenizer)
test_ds  = TextDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

Text Classifier (Model Definition)

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, num_classes, base_model):
        super().__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids,
                                  attention_mask=attention_mask)

        last_hidden = outputs.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()

        summed = torch.sum(last_hidden * mask_expanded, dim=1)
        summed_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        pooled = summed / summed_mask

        return self.fc(pooled)

Initialize Model

In [ ]:
model = TextClassifier(NUM_CLASSES, base_model).to(device)

# Freeze backbone initially
for param in model.base_model.parameters():
    param.requires_grad = False

Optimizer and Loss

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=INITIAL_LR
)

Training Loop 

In [ ]:
train_losses = []
val_losses = []
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    if epoch == UNFREEZE_EPOCH:
        print("Unfreezing DistilBERT backbone...")
        for param in model.base_model.parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=FINE_TUNE_LR
        )

        train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)

    # ---- Train ----
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # ---- Validation ----
    model.eval()
    val_loss_total = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            val_loss_total += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss_total / len(val_loader)
    val_acc = correct / total

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print("-" * 40)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print("Saved best model.")

Testing

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")